In [1]:
import os
import boto3
from sagemaker import get_execution_role
from pprint import pprint
import json
import time
import datetime as dt

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-04-30 20:46:03.486715


### Constants

In [3]:
# name of step function
str_name = 'genxi-payload-parsing'
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

Project: 20231010-gen-xii


### Write ```definition.json```

In [4]:
%%writefile definition.json

{
  "Comment": "A description of my state machine",
  "StartAt": "PullRequests",
  "States": {
    "PullRequests": {
      "Type": "Task",
      "Resource": "arn:aws:states:::batch:submitJob.sync",
      "Parameters": {
        "JobName": "job-def-genxi-pull-payloads-1",
        "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxi-pull-payloads-1:1",
        "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxi-pull-payloads-1"
      },
      "Next": "SplitRequests"
    },
    "SplitRequests": {
      "Type": "Task",
      "Resource": "arn:aws:states:::batch:submitJob.sync",
      "Parameters": {
        "JobName": "job-def-genxi-split-payloads-1",
        "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxi-split-payloads-1:3",
        "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxi-split-payloads-1"
      },
      "Next": "ParsePayloadsMap"
    },
    "ParsePayloadsMap": {
      "Type": "Map",
      "ItemProcessor": {
        "ProcessorConfig": {
          "Mode": "DISTRIBUTED",
          "ExecutionType": "STANDARD"
        },
        "StartAt": "ParsePayloads",
        "States": {
          "ParsePayloads": {
            "Type": "Task",
            "Resource": "arn:aws:states:::lambda:invoke",
            "OutputPath": "$.Payload",
            "Parameters": {
              "Payload.$": "$",
              "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxi-parse-payloads:$LATEST"
            },
            "Retry": [
              {
                "ErrorEquals": [
                  "Lambda.ServiceException",
                  "Lambda.AWSLambdaException",
                  "Lambda.SdkClientException",
                  "Lambda.TooManyRequestsException"
                ],
                "IntervalSeconds": 1,
                "MaxAttempts": 3,
                "BackoffRate": 2
              }
            ],
            "End": true
          }
        }
      },
      "ItemReader": {
        "Resource": "arn:aws:states:::s3:getObject",
        "ReaderConfig": {
          "InputType": "CSV",
          "CSVHeaderLocation": "FIRST_ROW"
        },
        "Parameters": {
          "Bucket": "20231010-gen-xii",
          "Key": "ad_hoc/gen_11_payload_parsing/df_idx.csv"
        }
      },
      "MaxConcurrency": 1000,
      "Label": "ParsePayloadsMap",
      "Next": "ConcatResults"
    },
    "ConcatResults": {
      "Type": "Task",
      "Resource": "arn:aws:states:::batch:submitJob.sync",
      "Parameters": {
        "JobName": "job-name-genxi-concat-parsed-payloads-1",
        "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxi-concat-parsed-payloads-1:1",
        "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxi-concat-parsed-payloads-1"
      },
      "End": true
    }
  }
}

Writing definition.json


### Make string definition

In [5]:
# load it
dict_definition = json.load(open('./definition.json'))
# make into string
str_definition = json.dumps(dict_definition)

### Create state machine

In [6]:
cls_client_sfn = boto3.client('stepfunctions')

In [7]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [8]:
# list state machines
dict_response = cls_client_sfn.list_state_machines(
)
list_dict_state_machines = dict_response['stateMachines']
list_dict_state_names = [{dict_state_machine['name']: dict_state_machine['stateMachineArn']} for dict_state_machine in list_dict_state_machines]
dict_state_names = {key: val for dict_name in list_dict_state_names for key, val in dict_name.items()}
pprint(dict_state_names)

{'MyStateMachine-fldl4s6of': 'arn:aws:states:us-west-2:836690756591:stateMachine:MyStateMachine-fldl4s6of',
 'gen-xi-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xi-retro-scoring',
 'gen-xii-payload-parsing-jq': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-payload-parsing-jq',
 'gen-xii-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-retro-scoring',
 'genxii-payload-parsing': 'arn:aws:states:us-west-2:836690756591:stateMachine:genxii-payload-parsing',
 'poc-step-genxii-lgd-lambda-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-lgd-lambda-boto3',
 'poc-step-genxii-pd-lambda-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-pd-lambda-boto3',
 'step-genxii-ad-feat-select-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-feat-select-boto3',
 'step-genxii-ad-model-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-model-

In [9]:
# get list of just names
list_str_names = [list(dict_state_names.keys())[0] for dict_state_names in list_dict_state_names]
# if our name is in there
if str_name in list_str_names:
    print(f'State machine {str_name} exists, it will be deleted')
    str_arn = dict_state_names[str_name]
    print(f'Deleting {str_arn}')
    print('')
    dict_response = cls_client_sfn.delete_state_machine(
        stateMachineArn=str_arn,
    )
    pprint(dict_response)
else:
    print(f'State machine {str_name} does not exist, so it will not be deleted')

State machine genxi-payload-parsing does not exist, so it will not be deleted


In [10]:
# make a state machine
while True:
    try:
        dict_response = cls_client_sfn.create_state_machine(
            name=str_name,
            definition=str_definition,
            roleArn=str_role,
            type='STANDARD',
        )
        pprint(dict_response)
        break
    except:
        pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '126',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Tue, 30 Apr 2024 20:46:04 GMT',
                                      'x-amzn-requestid': '9d424d99-8789-466d-9c65-3a0583d917ef'},
                      'HTTPStatusCode': 200,
                      'RequestId': '9d424d99-8789-466d-9c65-3a0583d917ef',
                      'RetryAttempts': 0},
 'creationDate': datetime.datetime(2024, 4, 30, 20, 46, 4, 102000, tzinfo=tzlocal()),
 'stateMachineArn': 'arn:aws:states:us-west-2:836690756591:stateMachine:genxi-payload-parsing'}


### Describe state machine

In [11]:
str_state_machine_arn = dict_response['stateMachineArn']
print(f'State Machine ARN: {str_state_machine_arn}')
dict_response = cls_client_sfn.describe_state_machine(
    stateMachineArn=str_state_machine_arn,
)
pprint(dict_response)

State Machine ARN: arn:aws:states:us-west-2:836690756591:stateMachine:genxi-payload-parsing
{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '2837',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Tue, 30 Apr 2024 20:46:04 GMT',
                                      'x-amzn-requestid': 'daa07053-4daf-4b40-ae30-213a70bc4878'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'daa07053-4daf-4b40-ae30-213a70bc4878',
                      'RetryAttempts': 0},
 'creationDate': datetime.datetime(2024, 4, 30, 20, 46, 4, 102000, tzinfo=tzlocal()),
 'definition': '{"Comment": "A description of my state machine", "StartAt": '
               '"PullRequests", "States": {"PullRequests": {"Type": "Task", '
               '"Resource": "arn:aws:states:::batch:submitJob.sync", '
               '"Parameters": {"JobName"

### Execute step function workflow

In [12]:
# # start execution
# dict_response = cls_client_sfn.start_execution(
#     stateMachineArn=str_state_machine_arn,
# )

### Clean-up

In [13]:
os.remove('./definition.json')